# Import packages

In [ ]:
import pandas as pd 
import geopandas as gpd # Working with maps
import matplotlib.pyplot as plt # Plotting
import contextily as ctx # For data about location (place names, roads, etc)

# Read in data

In [ ]:
# Extract sheet names; must read a particular sheet
xl = pd.ExcelFile('https://assets.publishing.service.gov.uk/media/691decfae39a085bda43efcd/File_2_IoD2025_Domains_of_Deprivation.xlsx')
xl.sheet_names

In [ ]:
# Read in IMD data
df = pd.read_excel('https://assets.publishing.service.gov.uk/media/691decfae39a085bda43efcd/File_2_IoD2025_Domains_of_Deprivation.xlsx', 
                  sheet_name = 'IoD2025 Domains')
df.shape # 33,755 rows of data, 20 columns

# Explore data

In [ ]:
df.columns # column names

In [ ]:
df # the data itself

In [ ]:
# Summary stats - like summary() in R
df.describe()

# Map data

In [ ]:
# Couldn't read map data in from the geoportal website, just downloaded it manually.
boundaries = gpd.read_file('/Users/joecrowley/Downloads/Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V5_-3761339731474027792.geojson')
# boundaries = gpd.read_file('https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V5/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson')
# boundaries = gpd.read_file('https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/LSOA_Dec_2021_Boundaries_Generalised_Clipped_BGC_EW_V2/FeatureServer/0/query?where=1%3D1&outFields=LSOA21CD&outSR=4326&f=geojson')
boundaries.shape

In [ ]:
boundaries

## Merge on IMD data

In [ ]:
# Merge boundaries with deprivation data
merged = boundaries.merge(df, left_on='LSOA21CD', right_on='LSOA code (2021)')

In [ ]:
merged

In [ ]:
df.columns

# Plot data

In [ ]:
# Plot - pick a deprivation column to shade by
merged.plot(column='Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)', cmap='Reds_r', figsize=(10, 12), legend=True)
plt.title('Income Deprivation by LSOA \n (Lower score = more deprived)')
plt.axis('off')
plt.show()

## Unique LAs in data

In [ ]:
# Show unique values for LA
for la in sorted(merged['Local Authority District name (2024)'].dropna().unique()):
    print(la)

<br> 

## Map Islington only

In [ ]:
# Select Islington data only
islington = merged[merged['Local Authority District name (2024)'] == 'Islington']
islington.shape

In [ ]:
# Plot Islington only
islington.plot(column='Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)', cmap='Reds_r', figsize=(10, 12), legend=True)
plt.title('Income Deprivation by LSOA')
plt.axis('off')
plt.show()

<br> 

### Adding context (1)

Converts the GeoDataFrame's coordinate reference system to EPSG:3857 (Web Mercator) — the standard projection used by web tile providers like Google Maps and OpenStreetMap. This is required so the data aligns correctly with the basemap.

In [ ]:
# Reproject to Web Mercator for web tiles
islington_web = islington.to_crs(epsg=3857)

<br> 

Creates a matplotlib figure (10×12 inches) with a single axes object `ax` that everything will be drawn on.

* `fig` (Figure) — the entire canvas/window, like a picture frame  
* `ax` (Axes) — the plot area inside that frame, where data, labels, and titles live  

Think of it like a whiteboard (`fig`) with a specific region marked out on it (`ax`).

`fig` is the top-level container — the entire blank canvas that holds everything.

<br> 

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))

* column — which data column to colour each polygon by (the IMD rank)  
* cmap='Reds_r' — uses a reversed red colour scale, so rank 1 (most deprived) appears darkest  
* legend=True — adds a colour bar showing the rank-to-colour mapping  
* alpha=0.5 — makes polygons 50% transparent so the basemap shows through  

In [ ]:
islington_web.plot(
    column='Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)',
    cmap='Reds_r',
    legend=True,
    alpha=0.5,
    ax=ax
)

Uses the contextily library to fetch and overlay a CartoDB Positron tile basemap — a clean, light-grey street map that contrasts well with the red choropleth.

In [ ]:
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

ax.set_title('Income Deprivation by LSOA (1 is most deprived)')
ax.axis('off')
plt.show()

<br>

### Adding context (2)

In [ ]:
# Reproject to Web Mercator for web tiles
islington_web = islington.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(10, 12))

islington_web.plot(
    column='Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)',
    cmap='Reds_r',
    legend=True,
    alpha=0.5,
    edgecolor='white',
    linewidth=0.2,
    ax=ax
)

# ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronOnlyLabels, zoom = 15)
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)

ax.set_title('Income Deprivation by LSOA (1 is most deprived)')
ax.axis('off')
plt.show()


# Presenting a filterable map

This works in the notebook, but not when presented in a quarto website, and is in both cases very slow. 

In [ ]:
# import ipywidgets as widgets
# from IPython.display import display, clear_output

# # ----------------------------
# # COLUMN NAMES
# # ----------------------------
# imd_col = "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)"
# la_col = "Local Authority District name (2024)"

# # ----------------------------
# # PREP DATA
# # ----------------------------
# gdf = merged.dropna(subset=[imd_col]).copy()

# # Reproject to WGS84 for interactive web maps
# gdf = gdf.to_crs(epsg=4326)

# # Simplified version for UK-wide view only
# gdf_proj = gdf.to_crs(epsg=3857)
# gdf_uk_simple = gdf_proj.copy()
# gdf_uk_simple["geometry"] = gdf_uk_simple.geometry.simplify(
#     tolerance=300,   # metres; try 50, 100, or 200
#     preserve_topology=True
# )
# gdf_uk_simple = gdf_uk_simple.to_crs(epsg=4326)

# # Fixed colour scale across all views
# vmin = gdf[imd_col].min()
# vmax = gdf[imd_col].max()

# # Dropdown options
# la_options = ["All UK"] + sorted(gdf[la_col].dropna().unique().tolist())

# dropdown = widgets.Dropdown(
#     options=la_options,
#     value="All UK",
#     description="Area:",
#     layout=widgets.Layout(width="500px")
# )

# out = widgets.Output()

# # ----------------------------
# # SINGLE VIEWER
# # ----------------------------
# def update_map(change=None):
#     with out:
#         clear_output(wait=True)

#         selected = dropdown.value

#         if selected == "All UK":
#             plot_gdf = gdf_uk_simple
#             m = plot_gdf.explore(
#                 column=imd_col,
#                 cmap="Reds_r",
#                 vmin=vmin,
#                 vmax=vmax,
#                 legend=True,
#                 tooltip=False,
#                 tiles="CartoDB positron",
#                 style_kwds={
#                     "weight": 0.05,
#                     "color": "white",
#                     "fillOpacity": 0.8
#                 },
#                 height=700,
#                 zoom_start=6, 
#                         map_kwds={
#             "zoomControl": False,
#             "scrollWheelZoom": False,
#             "doubleClickZoom": False,
#             "boxZoom": False,
#             "dragging": False
#         }
#             )
#         else:
#             plot_gdf = gdf[gdf[la_col] == selected]
#             m = plot_gdf.explore(
#                 column=imd_col,
#                 cmap="Reds_r",
#                 vmin=vmin,
#                 vmax=vmax,
#                 legend=True,
#                 tooltip=[la_col, imd_col],
#                 tiles="OpenStreetMap",
#                 style_kwds={
#                     "weight": 0.3,
#                     "color": "white",
#                     "fillOpacity": 0.6
#                 },
#                 height=700
#             )

#         display(m)

# dropdown.observe(update_map, names="value")

# display(dropdown)
# display(out)

# update_map()